In [1]:
# Required Libraries 

import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("✓ All Import Successful!")

✓ All Import Successful!


In [2]:
# Load environment variables


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")

print("API Key Loaded Successfully!")
print(f" Using Key: {OPENAI_API_KEY[:10]}...")

API Key Loaded Successfully!
 Using Key: sk-proj-Qc...


In [3]:


hr_policies_file = "./hr_policies.md"

if not os.path.exists(hr_policies_file):
    raise FileNotFoundError(f"{hr_policies_file} not found")

with open(hr_policies_file, "r", encoding="utf-8") as f:
    document_content = f.read()

    print(f" Document Loaded!")
    print(f"File size {len(document_content):,}characters")
    print(f"Lines: {len(document_content.splitlines()):,}")
    print(f"\n first 500 Characters:\n")
    print(document_content[:500] + "...")

 Document Loaded!
File size 13,070characters
Lines: 343

 first 500 Characters:

# HUMAN RESOURCES (HR) POLICY

**Organization Name:** Japanese XTY
**Effective Date:** [22/06/2025]
**Policy Owner:** Human Resources Department
**Approved By:** Management
**Version:** 1.0

## 1. Purpose

This HR Policy establishes clear guidelines for maintaining a professional, respectful, productive, healthy, and employee-friendly workplace. It defines the basic rules related to employment, working hours, attendance, leave, conduct, performance, workplace safety, employee well-being, and sep...


In [4]:

splitter = RecursiveCharacterTextSplitter(
separators=["\n\n","\n","."," "],
chunk_size = 600,
chunk_overlap=100,
length_function=len,
)

print(" RecursiveCharacterTextSplitter created with ")
print(" - chunk size: 500 characters")
print(" - overlap: 100 Characters")
print(r"- Separators: ['\n\n', '\n', '.', ' ']")

 RecursiveCharacterTextSplitter created with 
 - chunk size: 500 characters
 - overlap: 100 Characters
- Separators: ['\n\n', '\n', '.', ' ']


In [5]:
# splitting document


chunks = splitter.split_text(document_content)
print(f"Document Split Into{len(chunks)} chunks!\n")
print("Chunk Statistics:")
print(f" - Total chunks: {len(chunks)}")
print(f" - Averge chunk Size: {sum(len(c) for c in chunks)//len(chunks)} characters")
print(f" - Minimum chunk size: {min(len(c) for c in chunks)} characters")
print(f" - Maximum chunk size: {max(len(c) for c in chunks)} characters")

print(f"\n Example - Chunk 3: \n")
print(f"({len(chunks[2])} characters)")
print("-" * 60)
print(chunks[2])
print("-" * 60)

Document Split Into27 chunks!

Chunk Statistics:
 - Total chunks: 27
 - Averge chunk Size: 516 characters
 - Minimum chunk size: 322 characters
 - Maximum chunk size: 594 characters

 Example - Chunk 3: 

(588 characters)
------------------------------------------------------------
Discrimination or harassment contrary to applicable law or company policy will not be tolerated.

## 4. Recruitment and Onboarding

All recruitment shall be conducted through an approved hiring process.

New employees must:

* Submit required employment and identity documents.
* Complete background/reference verification where applicable.
* Sign the appointment/employment agreement.
* Complete joining formalities and orientation.
* Read and acknowledge applicable company policies.
* Maintain confidentiality regarding company and client information.

## 5. Probation and Confirmation
------------------------------------------------------------


In [6]:
# Create Embeddings

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    openai_api_key = OPENAI_API_KEY
)

print("OpenAI Embedding Initialized")
print(" Model: text-embedding-3-small")

sample_embedding = embeddings.embed_query(chunks[0])
print("Sample Embedding Created")
print(f"Vector Dimension: {len(sample_embedding)}")
print(f" First 10 Values: {sample_embedding[:10]}")


OpenAI Embedding Initialized
 Model: text-embedding-3-small
Sample Embedding Created
Vector Dimension: 1536
 First 10 Values: [0.01499176025390625, 0.0290374755859375, 0.0521240234375, 0.045806884765625, 0.02825927734375, -0.0307769775390625, -0.0200347900390625, -0.0193023681640625, 0.007801055908203125, -0.0016736984252929688]


In [8]:
# create chorma vector store


CHROMA_DB_PATH = "./chroma_db_notebook"

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DB_PATH,
    collection_name="hr_policies"
)

print("chroma vector database created")
print(f"Database path: {CHROMA_DB_PATH}")
print("Collection: hr_policies")
print(f"Chunk Stored:{len(chunks)}")
print(f" Vector Dimension: 1536")

chroma vector database created
Database path: ./chroma_db_notebook
Collection: hr_policies
Chunk Stored:27
 Vector Dimension: 1536


In [9]:
# create retriever


retriever = vectorstore.as_retriever(search_kwargs={"k":3})

# test query
test_query = "What is the nap policy"

print(f"Query: '{test_query}'\n")
print("="*70)

retrieved_docs = retriever.invoke(test_query)

print(f"\n Retrieved {len(retrieved_docs)} most similar chunks: \n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f" CHUNK {i}:")
    print("-" * 70)
    print(doc.page_content)
    print("-" * 70)
    print()




Query: 'What is the nap policy'


 Retrieved 3 most similar chunks: 

 CHUNK 1:
----------------------------------------------------------------------
### 9.4 Manager Approval and Team Coordination

Departments may establish a simple scheduling or approval process to ensure that adequate staff remain available.

Managers should administer the policy consistently and should not unreasonably deny permitted rest periods when business requirements allow them.

### 9.5 Misuse of Nap Policy

The nap policy is intended for short restorative breaks and should not be treated as additional recreational time.
----------------------------------------------------------------------

 CHUNK 2:
----------------------------------------------------------------------
### 9.4 Manager Approval and Team Coordination

Departments may establish a simple scheduling or approval process to ensure that adequate staff remain available.

Managers should administer the policy consistently and should not unreasonably

In [12]:
#create propmt templates

system_prompt = """You are a helpful HR assistant who knows all about Japanese XTY's HR policies. 
You provide accurate and detailed answers about the company's policies, even when they seem absurd. 
Always stay in character and be professional about these policies.

When answering, use the provided context from the HR policies document.
If the information is not in the policies, say so politely."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

print("Prompt Template Created")
print(f"System Prompt Length: {len(system_prompt)} characters")

Prompt Template Created
System Prompt Length: 380 characters


In [16]:
# LLM Initialize

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.7,
    openai_api_key=OPENAI_API_KEY
)

print("ChatOpenAI LLM Initialised")
print(f" Model: gpt-4.1-mini")
print(f"Temperature: 0.7 (Balanced Creativity)")

ChatOpenAI LLM Initialised
 Model: gpt-4.1-mini
Temperature: 0.7 (Balanced Creativity)


In [ ]:
# format retreived documents

